# **Brain Tumor Classification Demo Web App**

In [1]:
!pip install -q gradio tensorflow pillow numpy

In [2]:
# mount to drive:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import gradio as gr
import numpy as np
import tensorflow as tf
import time
from PIL import Image

# CONFIG
MODEL_PATH = "/content/drive/MyDrive/Brain_Tumor_Classification_Project/EFFNETB2.keras"
IMAGE_SIZE = 224
CLASSES = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]

# LOAD MODEL
model = tf.keras.models.load_model(MODEL_PATH, compile=False)

# PREPROCESS
def preprocess(img: Image.Image):
    img = img.convert("RGB")
    img = img.resize((IMAGE_SIZE, IMAGE_SIZE))
    return np.expand_dims(img, axis=0)

# PREDICT FUNCTION
def predict(img):
    start = time.time()
    x = preprocess(img)
    probs = model.predict(x, verbose=0)[0]
    elapsed = (time.time() - start) * 1000

    # For BAR (0–1)
    result = {CLASSES[i]: float(probs[i]) for i in range(len(CLASSES))}

    top_idx = int(np.argmax(probs))
    top_class = CLASSES[top_idx]

    # For TEXT (%)
    confidence = min(probs[top_idx] * 100, 99.99)

    return (
        f"{elapsed:.2f} ms",
        top_class,
        f"{confidence:.2f}%",
        result,
    )

# CUSTOM CSS
css = """
body {
    background: linear-gradient(120deg, #6a11cb, #2575fc);
}
.gr-box {
    border-radius: 20px !important;
}
"""

# UI
with gr.Blocks(css=css) as demo:
    gr.Markdown(
        """
        <h1 style='text-align:center;color:#00ff99'>Brain Tumor Classification</h1>
        <h3 style='text-align:center'>With EfficientNetB2 Architecture</h3>
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            img_input = gr.Image(type="pil", label="Input Image")

        with gr.Column(scale=2):
            time_txt = gr.Textbox(label="Time Inference", interactive=False)
            pred_label = gr.Textbox(label="Prediction", interactive=False)
            conf_txt = gr.Textbox(label="Confidence", interactive=False)
            prob_bar = gr.Label(num_top_classes=4, label="Class Probabilities")

    btn = gr.Button("Predict")
    btn.click(predict, img_input, [time_txt, pred_label, conf_txt, prob_bar])

    gr.Markdown(
        "<p style='text-align:center;color:#00ff99'>Developed by Tran Ky Dieu</p>"
    )

# RUN
demo.launch()


/tmp/ipykernel_669/644941293.py:55: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=css) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://813bff9248955a89a1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
